In [1]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

In [2]:
DATA_DIR = "/kaggle/input/datasets/hari31416/food-101/food-101"

MODERATE_CLASSES = [
    "beef_tartare",
    "chicken_quesadilla",
    "risotto",
    "spaghetti_carbonara",
    "pancakes"
]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),          
    transforms.RandomHorizontalFlip(),   
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


In [3]:
full_train = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
full_eval = datasets.ImageFolder(os.path.join(DATA_DIR, "validation"), transform=eval_transform)

class ModerateFoodDataset(Dataset):
    def __init__(self, full_dataset, class_names):
        self.dataset = full_dataset
        self.class_names = class_names
        self.valid_ids = [full_dataset.class_to_idx[name] for name in class_names]
        
        self.kept_indices = [i for i, (_, label) in enumerate(full_dataset.samples) if label in self.valid_ids]

    def __len__(self):
        return len(self.kept_indices)

    def __getitem__(self, idx):
        original_idx = self.kept_indices[idx]
        image, old_label = self.dataset[original_idx]
        new_label = self.valid_ids.index(old_label) 
        return image, new_label

train_dataset = ModerateFoodDataset(full_train, MODERATE_CLASSES)
eval_dataset = ModerateFoodDataset(full_eval, MODERATE_CLASSES)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(eval_dataset)}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/hari31416/food-101/food-101/train'